In [8]:
%pip install -q langchain langchain-community langchain-groq langchain-text-splitters python-dotenv chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_core.documents import Document
import os
from dotenv import load_dotenv

In [10]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key

gemini_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    os.environ["GEMINI_API_KEY"] = gemini_api_key

**Exemple of Documents**

In [11]:
# 1. Sample Wikipedia-style data
docs = [
    Document(page_content="Cats are small, carnivorous mammals that are often kept as pets. They typically live for 13-17 years."),
    Document(page_content="Dogs are domesticated mammals, not natural wild animals. They are known for their loyalty and live about 10-13 years."),
    Document(page_content="Elephants are the largest land animals. They communicate using low-frequency rumbles and can live for 60-70 years.")
]


**Splitting Text into chunks - Chunking**

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
split_docs = splitter.split_documents(docs)

**Generate Embeddings using pre-trained embedding model**

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\Speeker\AppData\Local\Temp\ipykernel_11404\1476618244.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4344.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Store Embeddings in chromaDB**

In [ ]:
vectorstore = Chroma.from_documents(split_docs, embedding_model, persist_directory="./rag_data")

In [15]:
!pip install langchain-google-genai

Defaulting to user installation because normal site-packages is not writeable


**LLM call**

In [25]:
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    max_output_tokens=512,
    api_key=gemini_api_key,
)
            

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


**RAG Pipeline (RAG Chain)**
  1. Retrieve relevant data
  2. Send it to the LLM
  3. Generate an answer

In [ ]:
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),  # searches in chromaDB and get top 2 relevant chunks
    return_source_documents=True
)

In [28]:
for q in questions:
    result = rag_chain({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")


Q: How do elephants communicate?
A: Elephants communicate using low-frequency rumbles.

Q: What is the average lifespan of a cat?
A: Cats typically live for 13-17 years.

Q: What are dogs known for?
A: Dogs are known for their loyalty.
